# 5.5 poskyris. Anglų ir lietuvių kalbos rezultatų palyginimas
## bei tarpkalbinis vertinimas

Analizuojami keturi eksperimentai:
- **EN → EN** – VoiceBank + DEMAND (tos pačios kalbos)
- **LT → LT** – sumažintas LIEPA-1 + DEMAND (tos pačios kalbos)
- **EN → LT** – VoiceBank modelis testuotas su LIEPA-1 aibe (tarpkalbinis)
- **LT → EN** – LIEPA-1 modelis testuotas su VoiceBank aibe (tarpkalbinis)

## 1. Importai ir nustatymai

In [ ]:
# 1. Importai ir nustatymai
import pandas as pd
import numpy as np
import os
import csv
import warnings
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        9,
    'axes.titlesize':   11,
    'axes.labelsize':   10,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
    'figure.dpi':       150,
})

CSV_PATH    = 'results/all_runs.csv'
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

# Keturi analizuojami eksperimentai (tikslūs pavadinimai)
EXPS = {
    'EN \u2192 EN': 'full_voicebank_unetdilatedmaskcnn_v2',
    'LT \u2192 LT': 'full_liepa_unetdilatedmaskcnn_v2',
    'EN \u2192 LT': 'cross_en_model_on_lt_unetdilatedmaskcnn_v2',
    'LT \u2192 EN': 'cross_lt_model_on_en_unetdilatedmaskcnn_v2',
}

LABELS_ORDER = ['EN \u2192 EN', 'LT \u2192 LT', 'EN \u2192 LT', 'LT \u2192 EN']

# Spalvos: tos pačios kalbos – mėlyna/žalia; tarpkalbiniai – oranžinė/violetinė
COLORS = {
    'EN \u2192 EN': '#2171B5',
    'LT \u2192 LT': '#41AB5D',
    'EN \u2192 LT': '#EF6C00',
    'LT \u2192 EN': '#8E24AA',
}

def lt_num(v, dec=3):
    return f'{v:.{dec}f}'.replace('.', ',')

print("Nustatymai įkelti.")
for lbl, exp in EXPS.items():
    print(f"  {lbl}  <-  {exp}")


## 2. Duomenų nuskaitymas ir diagnostika

In [ ]:
# 2. Duomenų nuskaitymas ir diagnostika
df = pd.read_csv(CSV_PATH)

print("Stulpeliai:")
print(list(df.columns))
print(f"\nIš viso eilučių: {len(df)}")
print()

target_ids = list(EXPS.values())
mask = df['experiment_id'].isin(target_ids)
print("=== Keturių eksperimentų eilutės ===")
display(df[mask][[
    'experiment_id', 'dataset_name', 'model_name',
    'num_eval_files', 'num_skipped_files',
    'mean_pesq_noisy', 'mean_pesq_enhanced', 'mean_delta_pesq',
    'mean_stoi_noisy', 'mean_stoi_enhanced', 'mean_delta_stoi',
    'mean_snr_noisy_est', 'mean_snr_enhanced_est', 'mean_delta_snr_est',
]])


## 3. Keturių eksperimentų radimas

In [ ]:
# 3. Keturiu eksperimentu radimas
results = {}

for label, exp_id in EXPS.items():
    rows = df[df['experiment_id'] == exp_id]
    if len(rows) == 0:
        print(f"ISPEJIMAS: '{exp_id}' NERASTAS!")
        results[label] = None
    else:
        if len(rows) > 1:
            print(f"  {exp_id}: rasta {len(rows)} eiluciu, naudojama paskutine.")
        results[label] = rows.iloc[-1]
        print(f"Rastas: {label}  <-  {exp_id}")

missing = [lbl for lbl, r in results.items() if r is None]
if missing:
    raise RuntimeError(f"Nerasti eksperimentai: {missing}")
print("\nVisi 4 eksperimentai rasti.")


## 4. Reikšmių ištraukimas ir patikra

In [ ]:
# 4. Reikšmių ištraukimas ir patikra
metrics = {}

for label in LABELS_ORDER:
    row = results[label]
    metrics[label] = {
        'pesq_before':  float(row['mean_pesq_noisy']),
        'pesq_after':   float(row['mean_pesq_enhanced']),
        'delta_pesq':   float(row['mean_delta_pesq']),
        'stoi_before':  float(row['mean_stoi_noisy']),
        'stoi_after':   float(row['mean_stoi_enhanced']),
        'delta_stoi':   float(row['mean_delta_stoi']),
        'snr_before':   float(row['mean_snr_noisy_est']),
        'snr_after':    float(row['mean_snr_enhanced_est']),
        'delta_snr':    float(row['mean_delta_snr_est']),
        'n_files':      int(row['num_eval_files']),
        'n_skipped':    int(row['num_skipped_files'])
                        if not pd.isna(row['num_skipped_files']) else 0,
        'train_data':   'VoiceBank + DEMAND' if 'voicebank' in str(row['experiment_id']) and 'cross' not in str(row['experiment_id'])
                        else ('LIEPA-1 + DEMAND (sumazintas)' if 'liepa' in str(row['experiment_id']) and 'cross' not in str(row['experiment_id'])
                        else ('VoiceBank + DEMAND' if 'cross_lt' in str(row['experiment_id'])
                        else 'LIEPA-1 + DEMAND (sumazintas)')),
        'test_data':    'VoiceBank + DEMAND' if row['dataset_name'] == 'voicebank'
                        else 'LIEPA-1 + DEMAND (sumazintas)',
        'vertinimas':   'Tos pacios kalbos' if label in ['EN \u2192 EN', 'LT \u2192 LT']
                        else 'Tarpkalbinis',
    }

print("=== Isrinkti duomenys ===")
for lbl in LABELS_ORDER:
    m = metrics[lbl]
    print(f"\n  {lbl} ({m['vertinimas']})")
    print(f"    DPESQ  = {lt_num(m['delta_pesq'])}")
    print(f"    DSTOI  = {lt_num(m['delta_stoi'])}")
    print(f"    DSNR   = {lt_num(m['delta_snr'], 2)} dB")
    print(f"    failai = {m['n_files']} (praleista: {m['n_skipped']})")


## 5. CSV lentelių išsaugojimas

In [ ]:
# 5. CSV lenteliu issaugojimas

# --- 5a. Bendra palyginimo lentelė ---
csv1 = os.path.join(FIGURES_DIR, 'kalbu_palyginimas_ir_tarpkalbinis_vertinimas.csv')
header1 = [
    'Vertinimas', 'Mokymo duomenys', 'Testavimo duomenys',
    'Testavimo failu skaicius', 'Pralestu failu skaicius',
    'PESQ pries', 'PESQ po', 'DPESQ',
    'STOI pries', 'STOI po', 'DSTOI',
    'SNR pries, dB', 'SNR po, dB', 'DSNR, dB',
]
rows1 = []
for lbl in LABELS_ORDER:
    m = metrics[lbl]
    rows1.append([
        lbl,
        m['train_data'],
        m['test_data'],
        str(m['n_files']),
        str(m['n_skipped']),
        lt_num(m['pesq_before'], 3),
        lt_num(m['pesq_after'],  3),
        lt_num(m['delta_pesq'],  3),
        lt_num(m['stoi_before'], 3),
        lt_num(m['stoi_after'],  3),
        lt_num(m['delta_stoi'],  3),
        lt_num(m['snr_before'],  2),
        lt_num(m['snr_after'],   2),
        lt_num(m['delta_snr'],   2),
    ])
with open(csv1, 'w', newline='', encoding='utf-8-sig') as f:
    csv.writer(f).writerows([header1] + rows1)
print(f"Issaugota: {csv1}")

# --- 5b. Tarpkalbinis vertinimas (tik EN→LT ir LT→EN) ---
csv2 = os.path.join(FIGURES_DIR, 'tarpkalbinis_vertinimas.csv')
header2 = [
    'Vertinimas', 'Mokymo duomenys', 'Testavimo duomenys',
    'PESQ pries', 'PESQ po', 'DPESQ',
    'STOI pries', 'STOI po', 'DSTOI',
    'SNR pries, dB', 'SNR po, dB', 'DSNR, dB',
]
rows2 = []
for lbl in ['EN \u2192 LT', 'LT \u2192 EN']:
    m = metrics[lbl]
    rows2.append([
        lbl, m['train_data'], m['test_data'],
        lt_num(m['pesq_before'], 3), lt_num(m['pesq_after'],  3), lt_num(m['delta_pesq'],  3),
        lt_num(m['stoi_before'], 3), lt_num(m['stoi_after'],  3), lt_num(m['delta_stoi'],  3),
        lt_num(m['snr_before'],  2), lt_num(m['snr_after'],   2), lt_num(m['delta_snr'],   2),
    ])
with open(csv2, 'w', newline='', encoding='utf-8-sig') as f:
    csv.writer(f).writerows([header2] + rows2)
print(f"Issaugota: {csv2}")

print()
display(pd.read_csv(csv1, encoding='utf-8-sig'))


## 6. ΔPESQ ir ΔSTOI palyginimas


In [ ]:
x      = np.arange(len(LABELS_ORDER))
bar_w  = 0.55
colors = [COLORS[lbl] for lbl in LABELS_ORDER]
edge   = '#333333'

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.8))

for ax, metric_key, ylabel, dec, title_txt in [
    (axes[0], 'delta_pesq', 'DPESQ', 3, 'DPESQ'),
    (axes[1], 'delta_stoi', 'DSTOI', 3, 'DSTOI'),
]:
    vals = [metrics[lbl][metric_key] for lbl in LABELS_ORDER]
    bars = ax.bar(x, vals, width=bar_w, color=colors,
                  edgecolor=edge, linewidth=0.8)

    ymax = max(max(vals) * 1.22, 0.001)
    ymin = min(0.0, min(vals) - abs(max(vals)) * 0.08)
    ax.set_ylim(ymin, ymax)

    pad = (ymax - ymin) * 0.015
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + pad if v >= 0 else bar.get_height() - pad * 3,
            lt_num(v, dec),
            ha='center', va='bottom' if v >= 0 else 'top',
            fontsize=8.5, fontweight='bold'
        )

    ax.set_xticks(x)
    ax.set_xticklabels(LABELS_ORDER, fontsize=8.5)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title_txt, fontsize=11, fontweight='bold', pad=6)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: lt_num(v, dec))
    )
    ax.axhline(0, color='#888888', linewidth=0.8, linestyle='-')
    ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.55)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Bendra legenda
legend_patches = [
    mpatches.Patch(facecolor=COLORS['EN \u2192 EN'], edgecolor=edge, label='EN \u2192 EN (tos pacios kalbos)'),
    mpatches.Patch(facecolor=COLORS['LT \u2192 LT'], edgecolor=edge, label='LT \u2192 LT (tos pacios kalbos)'),
    mpatches.Patch(facecolor=COLORS['EN \u2192 LT'], edgecolor=edge, label='EN \u2192 LT (tarpkalbinis)'),
    mpatches.Patch(facecolor=COLORS['LT \u2192 EN'], edgecolor=edge, label='LT \u2192 EN (tarpkalbinis)'),
]
fig.legend(handles=legend_patches, loc='lower center',
           bbox_to_anchor=(0.5, -0.04), ncol=2, fontsize=8,
           framealpha=0.9, edgecolor='#cccccc')

fig.suptitle(
    'Anglu ir lietuviu kalbos eksperimentu DPESQ ir DSTOI palyginimas',
    fontsize=10, fontweight='bold', y=1.01
)
plt.tight_layout(rect=[0, 0.10, 1, 1])

_png28 = os.path.join(FIGURES_DIR, '28_kalbu_palyginimas_delta_pesq_stoi.png')
_svg28 = os.path.join(FIGURES_DIR, '28_kalbu_palyginimas_delta_pesq_stoi.svg')
fig.savefig(_png28, dpi=300, bbox_inches='tight')
fig.savefig(_svg28, bbox_inches='tight')
plt.show()
print(f"Issaugota: {_png28}")
print(f"Issaugota: {_svg28}")


## 7. ΔSNR palyginimas


In [ ]:
vals   = [metrics[lbl]['delta_snr'] for lbl in LABELS_ORDER]
colors = [COLORS[lbl] for lbl in LABELS_ORDER]
x      = np.arange(len(LABELS_ORDER))
bar_w  = 0.55
edge   = '#333333'

fig, ax = plt.subplots(figsize=(6.5, 3.8))

bars = ax.bar(x, vals, width=bar_w, color=colors,
              edgecolor=edge, linewidth=0.8)

ymax = max(vals) * 1.20
ax.set_ylim(0, ymax)

pad = ymax * 0.015
for bar, v in zip(bars, vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + pad,
        lt_num(v, 2),
        ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

ax.set_xticks(x)
ax.set_xticklabels(LABELS_ORDER, fontsize=9)
ax.set_ylabel('DSNR, dB', fontsize=10)
ax.set_title(
    'Anglu ir lietuviu kalbos eksperimentu DSNR palyginimas',
    fontsize=10, fontweight='bold', pad=8
)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v, _: lt_num(v, 1))
)
ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.55)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend_patches = [
    mpatches.Patch(facecolor=COLORS['EN \u2192 EN'], edgecolor=edge, label='EN \u2192 EN (tos pacios kalbos)'),
    mpatches.Patch(facecolor=COLORS['LT \u2192 LT'], edgecolor=edge, label='LT \u2192 LT (tos pacios kalbos)'),
    mpatches.Patch(facecolor=COLORS['EN \u2192 LT'], edgecolor=edge, label='EN \u2192 LT (tarpkalbinis)'),
    mpatches.Patch(facecolor=COLORS['LT \u2192 EN'], edgecolor=edge, label='LT \u2192 EN (tarpkalbinis)'),
]
ax.legend(handles=legend_patches, loc='lower right',
          fontsize=8, framealpha=0.9, edgecolor='#cccccc')

plt.tight_layout()

_png29 = os.path.join(FIGURES_DIR, '29_kalbu_palyginimas_delta_snr.png')
_svg29 = os.path.join(FIGURES_DIR, '29_kalbu_palyginimas_delta_snr.svg')
fig.savefig(_png29, dpi=300, bbox_inches='tight')
fig.savefig(_svg29, bbox_inches='tight')
plt.show()
print(f"Issaugota: {_png29}")
print(f"Issaugota: {_svg29}")
